In [1]:
import sys
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, joblib, pyarrow.parquet as pq

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.features import feature_columns, to_matrix, ClippedScaler
from src.evaluation import evaluate
from src.pipeline import DATA, RESULTS, score_split

SEED = 11
torch.manual_seed(SEED); np.random.seed(SEED)

cols_all = feature_columns(pq.ParquetFile(DATA / "train.parquet").schema.names)
train = pd.read_parquet(DATA / "train.parquet", columns=cols_all).sample(1_000_000, random_state=SEED)

nz = (train > 0).mean()
cols = [c for c in cols_all if nz[c] >= 0.01]
print("kept", len(cols), "of", len(cols_all))
print("dropped:", [c for c in cols_all if nz[c] < 0.01])

kept 28 of 40
dropped: ['DURATION_IN', 'DURATION_OUT', 'RETRANSMITTED_IN_BYTES', 'RETRANSMITTED_IN_PKTS', 'RETRANSMITTED_OUT_BYTES', 'RETRANSMITTED_OUT_PKTS', 'NUM_PKTS_256_TO_512_BYTES', 'NUM_PKTS_512_TO_1024_BYTES', 'NUM_PKTS_1024_TO_1514_BYTES', 'ICMP_TYPE', 'ICMP_IPV4_TYPE', 'FTP_COMMAND_RET_CODE']


In [2]:
x_all = to_matrix(train, cols)
scaler = ClippedScaler(clip=5.0).fit(x_all)
joblib.dump({"cols": cols, "scaler": scaler}, RESULTS / "scaler_v2.joblib")

x_train = scaler.transform(x_all).astype(np.float32)
va = pd.read_parquet(DATA / "val.parquet", columns=cols).sample(100_000, random_state=SEED)
x_val = scaler.transform(to_matrix(va, cols)).astype(np.float32)
print(x_train.shape, x_val.shape)

(1000000, 28) (100000, 28)


In [3]:
d = x_train.shape[1]
model = nn.Sequential(
    nn.Linear(d, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, 12), nn.ReLU(),
    nn.Linear(12, 32), nn.ReLU(), nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, d))
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()
Xtr, Xva = torch.from_numpy(x_train), torch.from_numpy(x_val)

best, best_state, bad = float("inf"), None, 0
for epoch in range(30):
    model.train()
    perm = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr), 1024):
        xb = Xtr[perm[i:i + 1024]]
        opt.zero_grad(); loss_fn(model(xb), xb).backward(); opt.step()
    model.eval()
    with torch.no_grad():
        v = loss_fn(model(Xva), Xva).item()
    print(f"epoch {epoch + 1:2d}  val loss {v:.4f}")
    if v < best - 1e-4:
        best, bad = v, 0
        best_state = {k: t.clone() for k, t in model.state_dict().items()}
    else:
        bad += 1
        if bad >= 3:
            break
model.load_state_dict(best_state)

epoch  1  val loss 0.0253
epoch  2  val loss 0.0140
epoch  3  val loss 0.0104
epoch  4  val loss 0.0085
epoch  5  val loss 0.0069
epoch  6  val loss 0.0064
epoch  7  val loss 0.0053
epoch  8  val loss 0.0046
epoch  9  val loss 0.0041
epoch 10  val loss 0.0042
epoch 11  val loss 0.0037
epoch 12  val loss 0.0038
epoch 13  val loss 0.0034
epoch 14  val loss 0.0032
epoch 15  val loss 0.0033
epoch 16  val loss 0.0031
epoch 17  val loss 0.0030
epoch 18  val loss 0.0030
epoch 19  val loss 0.0028
epoch 20  val loss 0.0026
epoch 21  val loss 0.0026
epoch 22  val loss 0.0026
epoch 23  val loss 0.0028


<All keys matched successfully>

In [4]:
def ae_score(x):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(x), 100_000):
            xb = torch.from_numpy(x[i:i + 100_000])
            out.append(((model(xb) - xb) ** 2).mean(dim=1).numpy())
    return np.concatenate(out)

val_s = score_split("val", cols, scaler, ae_score)
tune_s = score_split("tune", cols, scaler, ae_score)
test_s = score_split("test", cols, scaler, ae_score, cap=100_000)

keys = ["auroc", "tpr@fpr0.01", "real_fpr@0.01", "tpr@fpr0.05", "real_fpr@0.05"]
for name, d_ in [("tune", tune_s), ("test", test_s)]:
    r = evaluate(d_, val_df=val_s)
    o = evaluate(d_)
    print(name, "val-threshold:", {k: round(r[k], 3) for k in keys})
    print(name, "ORACLE:       ", {k: round(o[k], 3) for k in ["auroc", "tpr@fpr0.01", "tpr@fpr0.05"]})
    print("   per-attack TPR@1%:", {k: round(v, 3) for k, v in r["per_attack_tpr@fpr0.01"].items()})

qs = [0.5, 0.9, 0.99, 0.999]
print(pd.DataFrame({"val_benign": val_s["score"].quantile(qs),
                    "test_benign": test_s[test_s["label"] == 0]["score"].quantile(qs)}))

pd.concat([val_s, tune_s, test_s]).to_parquet(
    RESULTS / "scores" / f"m3v2_autoencoder_nftoniot_seed{SEED}.parquet")

tune val-threshold: {'auroc': 0.953, 'tpr@fpr0.01': 0.007, 'real_fpr@0.01': 0.013, 'tpr@fpr0.05': 0.931, 'real_fpr@0.05': 0.084}
tune ORACLE:        {'auroc': 0.953, 'tpr@fpr0.01': 0.004, 'tpr@fpr0.05': 0.72}
   per-attack TPR@1%: {'ddos': 0.031, 'dos': 0.063, 'injection': 0.234, 'mitm': 0.0, 'password': 0.022, 'scanning': 0.003, 'xss': 0.09}
test val-threshold: {'auroc': 0.395, 'tpr@fpr0.01': 0.242, 'real_fpr@0.01': 0.536, 'tpr@fpr0.05': 0.923, 'real_fpr@0.05': 0.859}
test ORACLE:        {'auroc': 0.395, 'tpr@fpr0.01': 0.003, 'tpr@fpr0.05': 0.034}
   per-attack TPR@1%: {'backdoor': 0.998, 'ddos': 0.44, 'dos': 0.163, 'injection': 0.173, 'mitm': 0.321, 'password': 0.285, 'ransomware': 0.78, 'scanning': 0.102, 'xss': 0.139}
       val_benign  test_benign
0.500    0.000057     0.060792
0.900    0.001144     0.350547
0.990    0.049708     0.983652
0.999    0.226286     1.666568
